# Stuttering Detection: Multi-Model Comparative Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
---

## Step 1: Initialization

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import numpy as np
import torch.nn as nn
from src.data import DataManager
from src.models import (
    LogisticModel, NaiveBayesModel, ShallowNeuralNetwork, 
    DeepNeuralNetwork, KernelSVMModel, RandomForestModel
)

# Dataset Configuration (STRICT MODE)
SAMPLE_LIMIT = None
STRICT_LABELS = True
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2: Quality Filtering and Data Loading

In [2]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True, strict=STRICT_LABELS)
manager = DataManager(None, None)

# Enforcing full strict dataset load
X, y = manager.load_from_folders(fluent_dir, disfluent_dir, limit=SAMPLE_LIMIT, label_dict=label_dict)

print(f"Project Dataset: {len(X)} strict speech clips loaded.")
manager.analyze_distribution()

## Step 3: Run Multi-Model Comparison

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

X_train_f = manager.preprocess(X_train_bal, method="standard", fit=True)
X_test_f = manager.preprocess(X_test, method="standard", fit=False)

models = [
    LogisticModel("Logistic_Regression"),
    NaiveBayesModel("Naive_Bayes"),
    RandomForestModel("Random_Forest", n_estimators=100),
    KernelSVMModel("RBF_SVM", kernel='rbf')
]

results = []
for model in models:
    print(f"--- Training {model.name} ---")
    model.train(X_train_f, y_train_bal)
    res = model.evaluate(X_test_f, y_test)
    results.append((
        model.name, 
        res['accuracy'], 
        res['precision'], 
        res['recall'], 
        res['f1']
    ))

import pandas as pd
df_results = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1'])
print("\n--- Final Comparison Table (Strict Data) ---")
print(df_results.sort_values(by='Accuracy', ascending=False))